# Attention Sparsity Analysis

Analyzes how sparse or concentrated attention patterns are: entropy-based
sparsity, mass distribution, sparse vs dense heads, and attention windows.

**References:**
- Correia et al. (2019) "Adaptively Sparse Transformers"
- Shi et al. (2021) "Sparseformer: Sparse Attention for Visual Transformers"

## Why This Matters

Attention sparsity reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_sparsity import (
    attention_entropy_profile,
    attention_mass_distribution,
    sparse_vs_dense_heads,
    attention_window_analysis,
    attention_pattern_stability,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Attention entropy profile
ent = attention_entropy_profile(model, tokens)
print(f"Max possible entropy: {ent['max_possible_entropy']:.4f}")
print(f"Sparsest head: L{ent['sparsest_head'][0]}H{ent['sparsest_head'][1]}")
print(f"Densest head: L{ent['densest_head'][0]}H{ent['densest_head'][1]}")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        bar = '#' * int(ent['mean_entropy'][l, h] / ent['max_possible_entropy'] * 20)
        print(f"  L{l}H{h}: {ent['mean_entropy'][l, h]:.4f} {bar}")

In [ ]:
# 2. Attention mass distribution
mass = attention_mass_distribution(model, tokens, top_k=3)
print(f"Top-3 mass and Gini coefficient per head:")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        print(f"  L{l}H{h}: top3={mass['top_k_mass'][l,h]:.3f}, gini={mass['gini_coefficient'][l,h]:.3f}, "
              f"max={mass['max_attention'][l,h]:.3f}, eff_window={mass['effective_window'][l,h]:.1f}")

In [ ]:
# 3. Sparse vs dense head classification
sd = sparse_vs_dense_heads(model, tokens)
print(f"Sparse heads ({sd['n_sparse']}): {sd['sparse_heads']}")
print(f"Dense heads ({sd['n_dense']}): {sd['dense_heads']}")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        label = 'SPARSE' if sd['is_sparse'][l, h] else 'dense'
        bar = '#' * int(sd['sparsity_scores'][l, h] * 20)
        print(f"  L{l}H{h}: {sd['sparsity_scores'][l, h]:.3f} ({label}) {bar}")

In [ ]:
# 4. Attention window analysis
win = attention_window_analysis(model, tokens)
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        print(f"  L{l}H{h}: mean_dist={win['mean_distance'][l,h]:.2f}, "
              f"w90={win['window_90'][l,h]:.1f}, local={win['local_fraction'][l,h]:.3f}")

In [ ]:
# 5. Attention pattern stability across inputs
tokens_list = [
    jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45]),
    jnp.array([1, 6, 11, 16, 21, 26, 31, 36, 41, 46]),
    jnp.array([2, 7, 12, 17, 22, 27, 32, 37, 42, 47]),
]
stab = attention_pattern_stability(model, tokens_list)
print(f"Stable heads: {stab['stable_heads']}")
print(f"Unstable heads: {stab['unstable_heads']}")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        label = 'stable' if (l, h) in stab['stable_heads'] else 'UNSTABLE'
        print(f"  L{l}H{h}: var={stab['pattern_variance'][l,h]:.6f} ({label})")